In [1]:
# Our 7 target classes with consistent naming
CLASSES = {
    'Vitiligo': 0,
    'Melasma': 1,
    'Psoriasis': 2,
    'Eczema': 3,
    'Tinea': 4,
    'Contact Dermatitis': 5,
    'Seborrheic Dermatitis': 6
}

print(CLASSES)

{'Vitiligo': 0, 'Melasma': 1, 'Psoriasis': 2, 'Eczema': 3, 'Tinea': 4, 'Contact Dermatitis': 5, 'Seborrheic Dermatitis': 6}


What it does: Maps every DermaCon-IN disease label to one of our 7 classes, filters out everything irrelevant, and builds image paths

Why: Each dataset has its own naming conventions. This standardizes everything into our unified label system before combining

In [2]:
import pandas as pd
import os

# ── DermaCon-IN ──────────────────────────────────────────────────────
derma = pd.read_csv('../data/raw/DermaCon-IN/METADATA/Skin_Metadata-1.csv')

# Define which Disease_labels map to which of our 7 classes
derma_mapping = {
    'Vitiligo':               ['Vitiligo'],
    'Melasma':                ['Melasma'],
    'Psoriasis':              ['Psoriasis', 'Chronic plaque psoriasis', 'Guttate Psoriasis',
                               'Psoriasis Vulgaris', 'Palmar psoriasis', 
                               'Inverse psoriasis', 'Pustular psoriasis'],
    'Eczema':                 ['Eczema', 'Infected Eczema', 'Ear Eczema',
                               'Disseminated Eczema', 'Dry Discoid Eczema',
                               'Crusted eczematous dermatitis', 
                               'Chronic eczema with secondary infection'],
    'Tinea':                  ['Tinea Cruris', 'Tinea Corporis', 'Tinea Faciei',
                               'Steroid Modified Tinea', 'Tinea Versicolor',
                               'Tinea Capitis', 'Tinea', 'Tinea Manuum', 
                               'Tinea pedis', 'Infected Tinea'],
    'Contact Dermatitis':     ['Contact Dermatitis', 'Allergic Contact Dermatitis',
                               'Contact Dermatitis with secondary infection'],
    'Seborrheic Dermatitis':  ['Seborrheic Dermatitis'],
}

# Build a reverse lookup: Disease_label → our class name
reverse_mapping = {}
for class_name, labels in derma_mapping.items():
    for label in labels:
        reverse_mapping[label] = class_name

# Filter and remap
derma_filtered = derma[derma['Disease_label'].isin(reverse_mapping.keys())].copy()
derma_filtered['unified_label'] = derma_filtered['Disease_label'].map(reverse_mapping)
derma_filtered['numeric_label'] = derma_filtered['unified_label'].map(CLASSES)
derma_filtered['image_path'] = derma_filtered['Image_name'].apply(
    lambda x: f'../data/raw/DermaCon-IN/Dataset0/DATASET_0/{x}' 
    if os.path.exists(f'../data/raw/DermaCon-IN/Dataset0/DATASET_0/{x}') 
    else f'../data/raw/DermaCon-IN/Dataset1/DATASET_1/{x}'
)
print(f"DermaCon-IN filtered: {len(derma_filtered)} images")
print(derma_filtered['unified_label'].value_counts())

DermaCon-IN filtered: 2688 images
unified_label
Tinea                    1289
Vitiligo                  601
Eczema                    256
Psoriasis                 227
Contact Dermatitis        175
Melasma                    80
Seborrheic Dermatitis      60
Name: count, dtype: int64


What it does: SCIN stores labels as Python lists inside strings so we use ast.literal_eval to parse them properly, then take the top label and map to our 7 classes

Why: SCIN's label format is different from DermaCon-IN where each image has multiple possible labels with confidence scores, so we take the highest confidence one

In [4]:
import ast
import os

def extract_top_label(label_str):
    try:
        labels = ast.literal_eval(label_str)
        if isinstance(labels, list) and len(labels) > 0:
            return labels[0]
        return None
    except:
        return None

scin_mapping = {
    'Vitiligo':              ['Vitiligo'],
    'Melasma':               ['Melasma'],
    'Psoriasis':             ['Psoriasis'],
    'Eczema':                ['Eczema'],
    'Tinea':                 ['Tinea'],
    'Contact Dermatitis':    ['Contact Dermatitis', 'Allergic Contact Dermatitis'],
    'Seborrheic Dermatitis': ['Seborrheic Dermatitis'],
}

scin_reverse = {}
for class_name, labels in scin_mapping.items():
    for label in labels:
        scin_reverse[label] = class_name

scin = pd.read_csv('../data/raw/SCIN/scin_labels.csv', dtype={'case_id': str})
scin_cases = pd.read_csv('../data/raw/SCIN/scin_cases.csv', dtype={'case_id': str})

scin = scin.merge(scin_cases[['case_id', 'image_1_path']], on='case_id', how='left')
scin['top_label'] = scin['dermatologist_skin_condition_on_label_name'].apply(extract_top_label)

scin_filtered = scin[scin['top_label'].isin(scin_reverse.keys())].copy()
scin_filtered['unified_label'] = scin_filtered['top_label'].map(scin_reverse)
scin_filtered['numeric_label'] = scin_filtered['unified_label'].map(CLASSES)
scin_filtered['image_path'] = scin_filtered['image_1_path'].apply(
    lambda x: f'../data/raw/SCIN/images/{os.path.basename(x)}' if isinstance(x, str) else None
)

print(f"SCIN filtered: {len(scin_filtered)} images")
print(scin_filtered['unified_label'].value_counts())

for path in scin_filtered['image_path'].head(5):
    print(f"{path} — {'EXISTS' if os.path.exists(path) else 'MISSING'}")

SCIN filtered: 889 images
unified_label
Eczema                   470
Tinea                    154
Psoriasis                126
Contact Dermatitis       124
Seborrheic Dermatitis      7
Melasma                    5
Vitiligo                   3
Name: count, dtype: int64
../data/raw/SCIN/images/-217828380359571871.png — EXISTS
../data/raw/SCIN/images/-3060870142909393201.png — EXISTS
../data/raw/SCIN/images/-1306941150253534667.png — EXISTS
../data/raw/SCIN/images/-3933475004882152757.png — EXISTS
../data/raw/SCIN/images/-1979417173631887595.png — EXISTS


Numbers are lower than our earlier count because we're now only taking the top confidence label instead of counting any mention, as it is more accurate this way.

In [5]:
print(scin['case_id'].head(10))

0    -1000600354148496558
1    -1002039107727665188
2    -1003358831658393077
3    -1003826561155964328
4    -1003844406100696311
5    -1005079160214352144
6    -1010778459521153386
7    -1013831220015814987
8     -101827005996397499
9    -1022162013984621110
Name: case_id, dtype: str


In [6]:
# ── SkinDisNet ──────────────────────────────────────────────────────
skindiset = pd.read_csv('../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/SkinDisNet_Metadata.csv')

skindiset_mapping = {
    'Contact Dermatitis':    ['Contact Dermatitis'],
    'Eczema':                ['Eczema'],
    'Seborrheic Dermatitis': ['Seborrheic Dermatitis'],
    'Tinea':                 ['Tinea Corporis'],
}

skindiset_reverse = {}
for class_name, labels in skindiset_mapping.items():
    for label in labels:
        skindiset_reverse[label] = class_name

skindiset_filtered = skindiset[skindiset['Diagnosis'].isin(skindiset_reverse.keys())].copy()
skindiset_filtered['unified_label'] = skindiset_filtered['Diagnosis'].map(skindiset_reverse)
skindiset_filtered['numeric_label'] = skindiset_filtered['unified_label'].map(CLASSES)
skindiset_filtered['image_path'] = skindiset_filtered.apply(
    lambda row: f'../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/{row["Folder_name"]}/{row["Image_id"]}.jpg',
    axis=1
)

print(f"SkinDisNet filtered: {len(skindiset_filtered)} images")
print(skindiset_filtered['unified_label'].value_counts())

# Verify paths
for path in skindiset_filtered['image_path'].head(5):
    print(f"{path} — {'EXISTS' if os.path.exists(path) else 'MISSING'}")
    

SkinDisNet filtered: 1297 images
unified_label
Contact Dermatitis       477
Eczema                   466
Tinea                    275
Seborrheic Dermatitis     79
Name: count, dtype: int64
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (1).jpg — EXISTS
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (2).jpg — EXISTS
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (3).jpg — EXISTS
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (4).jpg — EXISTS
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (5).jpg — EXISTS


What it does: Loads SkinDisNet metadata, maps its folder-based labels to our 7 classes, and builds image paths using the Folder_name and Image_id columns

Why: SkinDisNet organizes images by condition folder (CD, EC, SD, TC) so we use both the folder name and image ID to construct the full path


In [13]:
# ── DermNet ────────────────────────────────────────────────────────────
import glob
import re

dermnet_base = '../data/raw/DermNet'

# Psoriasis - filter out histology, nails, penis
psoriasis_folder = f'{dermnet_base}/Psoriasis pictures Lichen Planus and related diseases'
psoriasis_files = [f for f in os.listdir(psoriasis_folder) 
                   if re.search(r'psoriasis', f, re.IGNORECASE)
                   and not re.search(r'histology|nails|penis', f, re.IGNORECASE)]

# Seborrheic Dermatitis
sebderm_files = [f for f in os.listdir(psoriasis_folder)
                 if re.search(r'seborrheic|sebderm', f, re.IGNORECASE)]

# Contact Dermatitis
contact_folder = f'{dermnet_base}/Poison Ivy Photos and other Contact Dermatitis'
contact_files = [f for f in os.listdir(contact_folder)
                 if re.search(r'contact.dermatitis|allergic.contact|irritant.contact|metal.dermatitis', f, re.IGNORECASE)]

# Eczema
eczema_folder = f'{dermnet_base}/Eczema Photos'
atopic_folder = f'{dermnet_base}/Atopic Dermatitis Photos'
eczema_files = os.listdir(eczema_folder)
atopic_files = os.listdir(atopic_folder)

# Build dataframe rows
dermnet_rows = []

for f in psoriasis_files:
    dermnet_rows.append({'image_path': f'{psoriasis_folder}/{f}', 'unified_label': 'Psoriasis', 'numeric_label': CLASSES['Psoriasis'], 'source': 'DermNet'})

for f in sebderm_files:
    dermnet_rows.append({'image_path': f'{psoriasis_folder}/{f}', 'unified_label': 'Seborrheic Dermatitis', 'numeric_label': CLASSES['Seborrheic Dermatitis'], 'source': 'DermNet'})

for f in contact_files:
    dermnet_rows.append({'image_path': f'{contact_folder}/{f}', 'unified_label': 'Contact Dermatitis', 'numeric_label': CLASSES['Contact Dermatitis'], 'source': 'DermNet'})

for f in eczema_files:
    dermnet_rows.append({'image_path': f'{eczema_folder}/{f}', 'unified_label': 'Eczema', 'numeric_label': CLASSES['Eczema'], 'source': 'DermNet'})

for f in atopic_files:
    dermnet_rows.append({'image_path': f'{atopic_folder}/{f}', 'unified_label': 'Eczema', 'numeric_label': CLASSES['Eczema'], 'source': 'DermNet'})

dermnet_df = pd.DataFrame(dermnet_rows)
print(f"DermNet filtered: {len(dermnet_df)} images")
print(dermnet_df['unified_label'].value_counts())

DermNet filtered: 625 images
unified_label
Eczema                   432
Psoriasis                141
Contact Dermatitis        27
Seborrheic Dermatitis     25
Name: count, dtype: int64


In [14]:
# ── SkinDiseaseImage ───────────────────────────────────────────────────
skinimg_base = '../data/raw/SkinDiseaseImage'

skinimg_mapping = {
    'Psoriasis': f'{skinimg_base}/7. Psoriasis pictures Lichen Planus and related diseases - 2k',
    'Eczema': f'{skinimg_base}/1. Eczema 1677',
    'Eczema2': f'{skinimg_base}/3. Atopic Dermatitis - 1.25k',
    'Tinea': f'{skinimg_base}/9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k',
}

skinimg_rows = []

for label, folder in skinimg_mapping.items():
    actual_label = 'Eczema' if label == 'Eczema2' else label
    for f in os.listdir(folder):
        if f.endswith('.jpg') or f.endswith('.png'):
            skinimg_rows.append({
                'image_path': f'{folder}/{f}',
                'unified_label': actual_label,
                'numeric_label': CLASSES[actual_label],
                'source': 'SkinDiseaseImage'
            })

skinimg_df = pd.DataFrame(skinimg_rows)
print(f"SkinDiseaseImage filtered: {len(skinimg_df)} images")
print(skinimg_df['unified_label'].value_counts())

SkinDiseaseImage filtered: 6691 images
unified_label
Eczema       2934
Psoriasis    2055
Tinea        1702
Name: count, dtype: int64


In [16]:
# ── Combine all datasets ────────────────────────────────────────────────
import pandas as pd

cols = ['image_path', 'unified_label', 'numeric_label']

derma_final = derma_filtered[cols].copy()
derma_final['source'] = 'DermaCon-IN'

scin_final = scin_filtered[cols].copy()
scin_final['source'] = 'SCIN'

skindiset_final = skindiset_filtered[cols].copy()
skindiset_final['source'] = 'SkinDisNet'

dermnet_final = dermnet_df[cols].copy()
dermnet_final['source'] = 'DermNet'

skinimg_final = skinimg_df[cols].copy()
skinimg_final['source'] = 'SkinDiseaseImage'

master_df = pd.concat([derma_final, scin_final, skindiset_final, dermnet_final, skinimg_final], ignore_index=True)

print(f"Total images: {len(master_df)}")
print(f"\nPer class:")
print(master_df['unified_label'].value_counts())
print(f"\nPer source:")
print(master_df['source'].value_counts())

Total images: 12190

Per class:
unified_label
Eczema                   4558
Tinea                    3420
Psoriasis                2549
Contact Dermatitis        803
Vitiligo                  604
Seborrheic Dermatitis     171
Melasma                    85
Name: count, dtype: int64

Per source:
source
SkinDiseaseImage    6691
DermaCon-IN         2688
SkinDisNet          1297
SCIN                 889
DermNet              625
Name: count, dtype: int64


What it does: Selects only the three columns we need from each dataset, adds a source column to track where each image came from, then concatenates all three into one master dataframe

Why: Having everything in one place makes it much easier to split into train/val/test and apply augmentation uniforml

In [17]:
import os
os.makedirs('../data/processed', exist_ok=True)
# Save master
master_df.to_csv('../data/processed/master_dataset.csv', index=False)
print("Saved master_dataset.csv")

Saved master_dataset.csv


In [18]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(
    master_df, 
    test_size=0.2, 
    random_state=42,
    stratify=master_df['numeric_label']
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,
    random_state=42,
    stratify=train_val_df['numeric_label']
)

print(f"Train: {len(train_df)} images")
print(f"Validation: {len(val_df)} images")
print(f"Test: {len(test_df)} images")
print(f"\nTrain class distribution:")
print(train_df['unified_label'].value_counts())

Train: 7801 images
Validation: 1951 images
Test: 2438 images

Train class distribution:
unified_label
Eczema                   2917
Tinea                    2189
Psoriasis                1631
Contact Dermatitis        514
Vitiligo                  386
Seborrheic Dermatitis     110
Melasma                    54
Name: count, dtype: int64


What it does: Splits data into 64% train, 16% validation, 20% test. stratify ensures each split has proportional representation of all 7 classes. random_state=42 makes the split reproducible.

Why: We need separate sets for training, tuning, and final evaluation. Test set is locked away and never touched until final evaluation. It simulates real world performance on unseen data

In [19]:
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

print("Saved:")
print(f"  train.csv — {len(train_df)} images")
print(f"  val.csv   — {len(val_df)} images")
print(f"  test.csv  — {len(test_df)} images")

Saved:
  train.csv — 7801 images
  val.csv   — 1951 images
  test.csv  — 2438 images
